# Fast DeltaNet kernels for Qwen3.8 27B: speed and parity check

Question: does installing `flash-linear-attention` replace the pure-torch gated delta rule that
dominates the 125 s per probe, and does it leave the logits unchanged?

The model is loaded once. The 48 Gated DeltaNet layers call a module-level chunk rule in
`transformers.models.qwen3_5.modeling_qwen3_5`, so the same weights are timed with the torch
fallback and then with the fla kernel swapped in for that name, on the same inputs. Nothing here touches probe data.

## Reading the result

- Parity passes when the max log-prob difference at the last position stays small and the top-1
  token is the same on every input.
- Speed is the ratio of the two forward times on a 7k-token input, one region read.
- Only if both hold is it worth re-timing a grid run with the fast kernel. Results from a
  different kernel are not pooled with the committed fallback runs without a parity check.

`causal-conv1d` is not installed: it needs a CUDA build, and the model falls back to a cheap
torch convolution without it. The chunk rule is the expensive part.


In [ ]:
import glob, os, subprocess, sys, zipfile

GIT_URL = 'https://github.com/rajul-kk/context-to-weights.git'
REPO = '/kaggle/working/myrios'
MARKER = 'baselines/cascading.py'
FRESH = True
REQUIRE = [('sleep/lm.py', 'chat_template_kwargs')]

def looks_like_source(d):
    return os.path.exists(os.path.join(d, MARKER))

os.makedirs('/kaggle/working', exist_ok=True)
os.chdir('/kaggle/working')

if FRESH and GIT_URL and os.path.exists(REPO):
    subprocess.run(['rm', '-rf', REPO], check=True)
    print(f'removed {REPO} so the clone is current')

if not looks_like_source(REPO) and GIT_URL:
    r = subprocess.run(['git', 'clone', '--depth', '1', GIT_URL, REPO],
                       capture_output=True, text=True)
    print(r.stdout, r.stderr)

if not looks_like_source(REPO):
    print('no clone; falling back to attached inputs')
    src = None
    for d in sorted(glob.glob('/kaggle/input/*')):
        if looks_like_source(d):
            src = d
            break
        for z in sorted(glob.glob(os.path.join(d, '*.zip'))):
            os.makedirs(REPO, exist_ok=True)
            zipfile.ZipFile(z).extractall(REPO)
            if looks_like_source(REPO):
                src = REPO
                break
        if src:
            break
    if src and src != REPO:
        subprocess.run(['cp', '-r', src, REPO], check=True)

assert looks_like_source(REPO), (
    'No source found. Either (a) set GIT_URL above, or (b) run scripts/package_source.py '
    'locally, upload the zip as a Kaggle Dataset, and attach it via Add Input. '
    f'Searched /kaggle/input/*, saw: {sorted(glob.glob("/kaggle/input/*"))}')

os.chdir(REPO)
sys.path.insert(0, REPO)

_h = subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True, cwd=REPO)
print('source commit:', _h.stdout.strip() or 'unknown (not a git checkout)')

_stale = [f for f, token in REQUIRE
          if not os.path.exists(f) or token not in open(f, encoding='utf-8').read()]
assert not _stale, (
    f'This checkout is older than the seed support these cells need: {_stale}. '
    'It came from an attached dataset or a leftover /kaggle/working/myrios rather than a '
    'fresh clone. Detach any source dataset under Add Input, keep only the runs_declare.zip '
    'dataset, and re-run this cell with FRESH = True.')
print('source is current for the seed grid')

subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'peft', 'accelerate', 'datasets'])

try:
    import importlib.metadata as _md
    _v = _md.version('torchao')
    if tuple(int(x) for x in _v.split('.')[:2]) < (0, 16):
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'uninstall', '-y', 'torchao'])
        print(f'removed incompatible torchao {_v}')
except Exception:
    pass

exec(open('notebooks/_runner.py').read())
print('cwd', os.getcwd())
print('files', sorted(os.listdir('.'))[:10])
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    print()
    print('=' * 68)
    print('NO GPU. Kaggle installed the CPU build of torch, so this session')
    print('has no accelerator attached. Everything below will be far too slow.')
    print()
    print('Fix: right panel -> Session options -> Accelerator -> GPU T4 x2,')
    print('then Run All again. The image swaps to a CUDA torch build on restart.')
    print('=' * 68)
else:
    print('gpu', torch.cuda.get_device_name(0),
          f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


Installs the kernels before any model code is imported.


In [ ]:
import os, sys, subprocess
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes>=0.46.1'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'flash-linear-attention', 'fla-core'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'list', '--format=freeze'], check=False,
               stdout=open('/kaggle/working/pip_freeze.txt', 'w'))
REPO = '/kaggle/working/myrios'
os.chdir(REPO); sys.path.insert(0, REPO)
if 'run' not in dir():
    exec(open('notebooks/_runner.py').read())

CFG = 'configs/kaggle_declare_27b.yaml'
MODEL = 'Qwen/Qwen3.8-27B'
from common.io import load_config
_c = load_config(CFG)
assert _c['model']['base'] == MODEL


## Loader


In [ ]:
import torch
from transformers import AutoConfig
from transformers.models.auto.modeling_auto import MODEL_FOR_CAUSAL_LM_MAPPING
assert torch.cuda.device_count() >= 2, 'need two T4s'
cfg_hf = AutoConfig.from_pretrained(MODEL)
LOADER = 'causal' if type(cfg_hf) in MODEL_FOR_CAUSAL_LM_MAPPING else 'image_text'
print('loader', LOADER)


## Check

Loads the weights once, then times and compares both chunk rules.


In [ ]:
CHECK_SRC = '''import sys, time, importlib, torch
sys.path.insert(0, '/kaggle/working/myrios')
import fla, triton
print('fla', getattr(fla, '__version__', '?'), 'triton', triton.__version__, 'torch', torch.__version__)
from common.io import load_config
from sleep.lm import load_backbone, chat_text
from declare.elicit import register_repeat

cfg = load_config(sys.argv[1], {'model': {'loader': sys.argv[2]}})
model, tokenizer = load_backbone(cfg, attn_implementation=register_repeat())

mod = sys.modules['transformers.models.qwen3_5.modeling_qwen3_5']
from fla.ops.gated_delta_rule import chunk_gated_delta_rule
slow = mod.torch_chunk_gated_delta_rule
calls = {'fast': 0}

def fast(*args, chunk_size=None, **kwargs):
    calls['fast'] += 1
    return chunk_gated_delta_rule(*args, **kwargs)

dev = next(model.parameters()).device
base = chat_text(tokenizer, None, 'Answer with only Yes or No. Is Paris the capital of France?')
filler = ' '.join(f'Item {i} is stored in box {i * 7 % 101}.' for i in range(1200))
long_text = chat_text(tokenizer, None, filler + ' Answer with only Yes or No. Is box 7 listed?')
inputs = {}
for name, text, cap in (('short', base, None), ('long', long_text, 7000)):
    ids = tokenizer([text], return_tensors='pt')['input_ids'][:, -cap:] if cap else tokenizer([text], return_tensors='pt')['input_ids']
    inputs[name] = ids.to(dev)
    print(name, 'tokens', ids.shape[1])

def forward(ids):
    torch.cuda.synchronize()
    t = time.time()
    with torch.no_grad():
        out = model(input_ids=ids, logits_to_keep=1).logits[0, -1].float()
    torch.cuda.synchronize()
    return torch.log_softmax(out, -1).cpu(), time.time() - t

def set_rule(rule):
    mod.torch_chunk_gated_delta_rule = rule

results = {}
for label, rule in (('torch', slow), ('fast', fast)):
    set_rule(rule)
    forward(inputs['short'])
    for name in inputs:
        lp, secs = forward(inputs[name])
        results[(label, name)] = (lp, secs)
        if label == 'fast':
            assert calls['fast'] % 48 == 0 and calls['fast'] > 0, f"fast rule called {calls['fast']} times, expected a multiple of 48"
        else:
            assert calls['fast'] == 0, 'fast rule ran during the torch pass'
        peaks = [round(torch.cuda.max_memory_allocated(i) / 2**30, 2) for i in range(torch.cuda.device_count())]
        print(f'{label:5s} {name:5s} {secs:7.2f} s  peak GiB {peaks}', flush=True)
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()

ok = True
for name in inputs:
    a, b = results[('torch', name)][0], results[('fast', name)][0]
    diff = (a - b).abs().max().item()
    same = a.argmax().item() == b.argmax().item()
    ratio = results[('torch', name)][1] / results[('fast', name)][1]
    print(f'{name}: max logprob diff {diff:.4f}  same top-1 {same}  speedup {ratio:.1f}x  top token {tokenizer.decode(a.argmax().item())!r}')
    ok = ok and same and diff < 0.25
print('PARITY', 'PASS' if ok else 'FAIL')
'''
open('/kaggle/working/fk_check.py', 'w').write(CHECK_SRC)
run(f'python /kaggle/working/fk_check.py {CFG} {LOADER}')
